# UR5e VLA bed — training run (Kaggle, free)
One run per session. Parameters come from the smoke's projection table; **Save & Run All (Commit)** so it keeps running with the tab closed. Inputs: dataset **vla-bed-v2**; Accelerator GPU T4 x2; Internet ON. Trains, checks the last checkpoint on 50 episodes, packs all checkpoints; the full suite runs in `eval.ipynb`.

In [ ]:
# Filled from smoke v5's projection table (Tesla T4, fp16-b32: 0.763 steps/s → 10k steps ≈ 3.6 h). One run per session.
RUN = "baseline"          # baseline | gripper | chunkwise | plastic
RECIPE = "auto"           # "auto" = the recipe of the single vla-bed-<recipe> dataset attached (v2, or v3 = headroom 0.7 + camera jitter); or name it
STEPS = 10000
BATCH = 32
VLM_DTYPE = "float16"     # smoke v4/v5 (T4): float16 0.76 steps/s vs bfloat16 0.21 — the T4 has no native bfloat16
SAVE_EVERY = 2500
MAX_HOURS = 6.0           # training budget; the rest of the 8 h is for the quick check + packing
QUICK_EVAL_EPISODES = 50  # final checkpoint only, nominal; MuJoCo on Kaggle's CPU costs ~66 s per 100-frame episode (smoke v5), so the full suite runs in eval.ipynb

In [ ]:
import os, subprocess, sys, time, json, pathlib
REPO = "https://github.com/santapong/RoboLLM.git"; BRANCH = "experiment/ur5e-vla-bed"
ROOT = pathlib.Path("/kaggle/working/RoboLLM")
# Kaggle mounts the uploaded zip under /kaggle/input/<slug>/ with or without the zip's top folder; find the manifest.
RECIPE = globals().get("RECIPE", "auto")
# A bundle uploaded as split parts (vla-bed-<recipe>.zip.partNN + SHA256SUMS, made by split -b 9M) is joined and unpacked under /tmp first.
import hashlib, zipfile
for sums in pathlib.Path("/kaggle/input").rglob("SHA256SUMS"):
    parts = sorted(sums.parent.glob("*.zip.part*"))
    if not parts: continue
    name = parts[0].name.split(".zip.part")[0]; joined = pathlib.Path("/tmp/bundles") / f"{name}.zip"; joined.parent.mkdir(parents=True, exist_ok=True)
    if not joined.exists():
        with open(joined, "wb") as out:
            for part in parts: out.write(part.read_bytes())
    want = next((line.split()[0] for line in sums.read_text().splitlines() if line.strip().endswith(f" {name}.zip")), None)
    got = hashlib.sha256(joined.read_bytes()).hexdigest()
    assert want is None or got == want, f"{name}.zip sha256 {got} != {want} (parts incomplete?)"
    dest = pathlib.Path("/tmp/bundles") / name
    if not dest.exists(): zipfile.ZipFile(joined).extractall(dest)
    print("joined", len(parts), "parts →", dest, "sha256 OK" if want else "sha256 unchecked")
roots = [pathlib.Path("/kaggle/input"), pathlib.Path("/tmp/bundles")]
found = {json.load(open(p)).get("recipe"): p.parent for r in roots if r.exists() for p in r.rglob("manifest.json") if (p.parent / "train").is_dir()}
if RECIPE == "auto":   # exactly one bed dataset attached → its recipe
    assert len(found) == 1, "attach exactly one vla-bed-<recipe> dataset or set RECIPE; found: " + str(found)
    RECIPE = next(iter(found))
assert RECIPE in found, f"add the private dataset vla-bed-{RECIPE} to this notebook (Add Input); found: " + str(found)
DATA = found[RECIPE]; print("dataset root", DATA, "recipe", RECIPE)
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(ROOT)], check=True)
os.chdir(ROOT)
link = ROOT / "datasets" / "vla-bed" / RECIPE; link.parent.mkdir(parents=True, exist_ok=True)
MANIFEST = f"datasets/vla-bed/{RECIPE}/manifest.json"   # the frozen suite: v3 records the same 100 evaluation seeds/targets as v2
if not link.exists(): link.symlink_to(DATA)          # every default path in the bed now resolves to the uploaded data
MEN = ROOT / "sim" / "vla-bed" / "assets" / "mujoco_menagerie"   # robot models are not vendored (BSD notices in NOTICES.md); pinned sparse clone, as scripts/pi_setup.sh does
if not (MEN / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--filter=blob:none", "--no-checkout", "https://github.com/google-deepmind/mujoco_menagerie.git", str(MEN)], check=True)
    subprocess.run(["git", "-C", str(MEN), "sparse-checkout", "set", "universal_robots_ur5e", "robotiq_2f85"], check=True)
subprocess.run(["git", "-C", str(MEN), "checkout", "--quiet", "e4049d0a3bfd58d2a3081614e6777d4007e3f86a"], check=True)
print("menagerie", subprocess.run(["git", "-C", str(MEN), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "sim/vla-bed/requirements-record.txt", "mujoco==3.10.0", "pyyaml", "av"], check=True)  # the bed's physics is not in LeRobot's extras
subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1 || true", shell=True)   # MuJoCo fallback renderer
os.environ["MUJOCO_GL"] = "egl"; os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "bf16 native", torch.cuda.is_bf16_supported())
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Which renderer works here? EGL (NVIDIA) first, OSMesa second.
import os, subprocess, sys
def try_gl(backend):
    r = subprocess.run([sys.executable, "-c", "import mujoco,numpy as np; m=mujoco.MjModel.from_xml_string('<mujoco><worldbody><geom size=\"1\"/></worldbody></mujoco>'); d=mujoco.MjData(m); r=mujoco.Renderer(m,64,64); r.update_scene(d); print(r.render().mean())"], env={**os.environ, "MUJOCO_GL": backend}, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()[-200:]
for b in ("egl", "osmesa"):
    ok, msg = try_gl(b); print(b, "OK" if ok else "FAIL", msg if not ok else "")
    if ok: os.environ["MUJOCO_GL"] = b; break
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "sim/vla-bed/gpu/preflight.py", "--execute", "--dataset-root", f"datasets/vla-bed/{RECIPE}", "--min-disk-gb", "10"], capture_output=True, text=True).stdout[-1500:])

In [ ]:
# DAgger (recipe v7): if a rollout zip from dagger.ipynb is attached next to the base dataset, merge base ∪ rollouts and train on the union.
import glob, zipfile, pathlib, subprocess, sys, json, os
v7zips = glob.glob("/kaggle/input/**/vla-bed-v7.zip", recursive=True)
DATASET_ROOT = f"datasets/vla-bed/{RECIPE}/train"
if v7zips:
    zipfile.ZipFile(v7zips[0]).extractall("/tmp/dagger")
    link7 = ROOT / "datasets" / "vla-bed" / "v7"
    if not link7.exists(): link7.symlink_to(pathlib.Path("/tmp/dagger/v7"))
    r = subprocess.run([sys.executable, "sim/vla-bed/dagger.py", "merge", "--base", RECIPE], capture_output=True, text=True); print(r.stdout[-800:], r.stderr[-800:] if r.returncode else "")
    assert r.returncode == 0, "merge failed"
    DATASET_ROOT = "datasets/vla-bed/v7/train-merged"; RUN_TAG = f"{RUN}-v7"
    print("training on", DATASET_ROOT, "(base", RECIPE, "∪ DAgger rollouts)", json.load(open("/tmp/dagger/v7/manifest.json"))["relabel"])
else:
    RUN_TAG = RUN
print("DATASET_ROOT =", DATASET_ROOT)

In [ ]:
import subprocess, sys, json, pathlib
cmd = [sys.executable, "sim/vla-bed/gpu/train.py", "--run", RUN, "--mode", "full", "--steps", str(STEPS), "--batch-size", str(BATCH), "--save-freq", str(SAVE_EVERY), "--max-hours", str(MAX_HOURS), "--vlm-dtype", VLM_DTYPE, "--dataset-root", DATASET_ROOT, "--execute"]
print(" ".join(cmd))
r = subprocess.run(cmd)                       # streams LeRobot's own log; checkpoints under artifacts/vla-bed/<RUN>/full/checkpoints/
rec = pathlib.Path(f"artifacts/vla-bed/{RUN}/full/run_record.json")
print(json.dumps({k: v for k, v in json.loads(rec.read_text()).items() if k in ("status", "steps_done", "steps_per_s", "wall_s", "gpu", "peak_vram_gb")}, indent=1) if rec.exists() else f"no run record (rc={r.returncode})")

In [ ]:
# Quick closed-loop check of the LAST checkpoint only (nominal, QUICK_EVAL_EPISODES episodes); the full suite is eval.ipynb's job.
import subprocess, sys, glob, os
cks = sorted(glob.glob(f"artifacts/vla-bed/{RUN}/full/checkpoints/*/pretrained_model"))
print("checkpoints:", [c.split("/")[-2] for c in cks])
last = cks[-1]; step = last.split("/")[-2]
r = subprocess.run([sys.executable, "sim/vla-bed/evaluate.py", "--policy", "smolvla", "--run", RUN, "--checkpoint", last, "--episodes", str(QUICK_EVAL_EPISODES), "--label", f"{RUN}/{step}", "--variation", "nominal", "--manifest", MANIFEST], capture_output=True, text=True, env={**os.environ, "MUJOCO_GL": os.environ.get("MUJOCO_GL", "egl")})
print(r.stdout[-1500:], r.stderr[-600:] if r.returncode else "")

In [ ]:
# Pack results + ALL checkpoints (≈ 865 MB each; 4 × 10k/2.5k ≈ 3.5 GB, under Kaggle's 20 GB output cap) into /kaggle/working.
import json, glob, shutil, pathlib, hashlib, socket
host = socket.gethostname()
stage = pathlib.Path("/kaggle/working/pack"); shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(f"sim/vla-bed/results/p5/{host}", stage / "results" / "p5" / host)
for ck in sorted(glob.glob(f"artifacts/vla-bed/{RUN}/full/checkpoints/*/pretrained_model")):
    step = ck.split("/")[-2]; shutil.copytree(ck, stage / "artifacts" / RUN / "kaggle" / step / "pretrained_model")
shutil.copy(f"artifacts/vla-bed/{RUN}/full/run_record.json", stage / "run_record.json")
(stage / "recipe.txt").write_text(RECIPE)
out = shutil.make_archive(f"/kaggle/working/vla-bed-{globals().get('RUN_TAG', RUN)}-output" + ("" if RECIPE == "v2" else f"-{RECIPE}"), "zip", stage)
shutil.rmtree(stage); shutil.rmtree(f"artifacts/vla-bed/{RUN}/full/checkpoints", ignore_errors=True)   # keep the output tab small: the zip is the deliverable
print(out, round(pathlib.Path(out).stat().st_size / 1e6), "MB", "sha256", hashlib.sha256(open(out, "rb").read()).hexdigest())
print("next: eval.ipynb with this notebook's output as an input; on the workstation: sim/vla-bed/gpu/kaggle_import.sh <zip> <sha256>")